**# En este archivo se hara la limpieza y transformacion de los datos de la microbioma intestinal**

In [52]:
import os
os.listdir('/kaggle/input/datasets/nico14h/metagenomica')

['SraRunTable(2).csv',
 'diagnosticos_pacientes.tsv',
 'matriz_final.tsv',
 'matriztotal.tsv',
 'taxonomy.tsv',
 'dataset.csv']

In [53]:
#pip install scikit-bio

In [54]:
import pandas as pd
import numpy as np


#tenemos que en matriz, cada una de las columnas son los pacientes, y las filas son los hash
matriz= pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/matriz_final.tsv', sep = '\t', skiprows=1)
matriz= matriz.rename(columns={'#OTU ID': 'Feature ID'})
taxonomia = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/taxonomy.tsv', sep='\t')
#en taxonomia, tendremos: fila(hash) | taxonomia | confianza( de que sea esa bacteria)
# tenemos que los hash NO SE REPITEN, son UNICOS, no hay filas repetidas

#comienzo eliminando lo innecesario de la 3era columna ya que solo me importa el genero de la bacteria, no toda
#la informacion adicional
#ATENCION: es recomendable realizar un analisis con especies de igual manera, no solo genero
def extraer_nombre_bacteria(tax_string):
    # Si la celda está vacía o sin asignar
    if pd.isna(tax_string) or tax_string == 'Unassigned':
        return 'Desconocida'

    # Cortamos el string en una lista de niveles
    niveles = tax_string.split(';')

    # 1. EL FILTRO: Eliminamos por completo la especie ('s__') si es que existe
    niveles_sin_especie = [nivel for nivel in niveles if not nivel.strip().startswith('s__')]

    # 2. BÚSQUEDA: Recorremos lo que quedó de atrás para adelante
    for nivel in reversed(niveles_sin_especie):
        nivel_limpio = nivel.strip()

        # Buscamos el separador biológico
        if '__' in nivel_limpio:
            nombre_puro = nivel_limpio.split('__')[-1].strip()

            # Si el nivel no está vacío (ej. no es solo "g__"), lo retornamos
            if nombre_puro:
                return nombre_puro

    # Fallback de seguridad
    return 'Desconocida'


taxonomia['Bacteria_Limpia'] = taxonomia['Taxon'].apply(extraer_nombre_bacteria)

#luego hago un inner join, usando los hash

df_fusionado = pd.merge(taxonomia[['Feature ID', 'Bacteria_Limpia']], matriz, on='Feature ID', how='inner')

#elimino el hash de cada caso
df_fusionado = df_fusionado.drop(columns=['Feature ID'])

# Agrupamos por el nombre de la bacteria y sumamos los valores numéricos de los pacientes
df_fase1 = df_fusionado.groupby('Bacteria_Limpia').sum().reset_index()

# Verificamos el resultado final
print("Dimensiones de la matriz agrupada:", df_fase1.shape)
print(df_fase1.head())
# se crea el archivo matriztotal.tsv, no se crea en kaggle ya que solo es lectura
#df_fase1.to_csv('/kaggle/input/datasets/nico14h/metagenomica/matriztotal.tsv', sep='\t', index=False)

Dimensiones de la matriz agrupada: (403, 122)
  Bacteria_Limpia  SRR10442269_1  SRR10442270_1  SRR10442271_1  SRR10442272_1  \
0           11-24            0.0            0.0            0.0            0.0   
1          4-29-1            0.0            0.0            0.0            0.0   
2              A2            0.0            0.0            0.0            0.0   
3  ADurb.Bin063-1            0.0            0.0            0.0            0.0   
4        AKAU4049            0.0            0.0            0.0            0.0   

   SRR10442273_1  SRR10442274_1  SRR10442275_1  SRR10442276_1  SRR10442277_1  \
0            0.0            0.0            0.0            0.0            0.0   
1            0.0            0.0            0.0            0.0            0.0   
2            0.0            0.0            0.0            0.0            0.0   
3            0.0            0.0            0.0            0.0            0.0   
4            0.0            0.0            0.0            0.0      

In [55]:
import pandas as pd

# 1. Identificamos cuáles son las columnas de los pacientes 
# (Básicamente, todas las columnas excepto 'Bacteria_Limpia')
columnas_pacientes = df_fase1.columns.drop('Bacteria_Limpia')

# 2. Aislamos SOLO la parte de los números para hacer el cálculo
# Esto crea una vista temporal sin la columna de texto
df_solo_numeros = df_fase1[columnas_pacientes]

# 3. Calculamos la prevalencia FILA por FILA (axis=1)
# (df_solo_numeros > 0) genera True si el valor es mayor a 0
# .mean(axis=1) saca el porcentaje de "True" para cada bacteria (fila)
prevalencia = (df_solo_numeros > 0).mean(axis=1)

# 4. Filtramos el DATAFRAME ORIGINAL
# Le decimos a Pandas: "Del df_fase1 original, guardame solo las filas 
# donde la prevalencia calculada sea mayor o igual a 0.10"
umbral = 0.05
df_filtrado = df_fase1[prevalencia >= umbral]

# Verificamos los resultados
print(f"Cantidad de bacterias antes del filtro: {df_fase1.shape[0]}")
print(f"Cantidad de bacterias después del filtro: {df_filtrado.shape[0]}")
print("\nMuestra del DataFrame limpio:")
print(df_filtrado['Bacteria_Limpia'])


Cantidad de bacterias antes del filtro: 403
Cantidad de bacterias después del filtro: 153

Muestra del DataFrame limpio:
5                           Abiotrophia
13                        Acinetobacter
16                          Actinomyces
17                        Adlercreutzia
19                         Agathobacter
                     ...               
395    [Eubacterium]_xylanophilum_group
396     [Ruminococcus]_gauvreauii_group
397         [Ruminococcus]_gnavus_group
398        [Ruminococcus]_torques_group
402                          uncultured
Name: Bacteria_Limpia, Length: 153, dtype: object


In [56]:
generos_a_eliminar = [
    'Ralstonia', 'Burkholderia', 'Cupriavidus', 'Methylobacterium', 
    'Bradyrhizobium', 'Rhizobium', 'Sphingomonas', 'Novosphingobium',
    'Acetobacter', 'Gluconobacter', 'Oenococcus', 'Cutibacterium'
]

# df_filtrado es tu matriz de 133 bacterias
# Vemos cuántas de las bacterias actuales están en la lista de eliminables
en_la_lista = df_filtrado.index[df_filtrado.index.isin(generos_a_eliminar)]

print(f"De las 133 bacterias, solo {len(en_la_lista)} son contaminantes de tu lista.")
print("Bacterias que se van a borrar:", list(en_la_lista))

De las 133 bacterias, solo 0 son contaminantes de tu lista.
Bacterias que se van a borrar: []


In [57]:
from skbio.stats.composition import multi_replace, clr

# 1. Extraemos solo los nombres de los pacientes (las columnas numéricas)
df_purificado = df_filtrado
df_purificado = df_purificado.set_index('Bacteria_Limpia')
columnas_pacientes = df_purificado.columns.drop('Bacteria_Limpia') if 'Bacteria_Limpia' in df_purificado.columns else df_purificado.columns

# 2. Aislamos los datos numéricos y los Transponemos (.T)
matriz_cruda_transpuesta = df_purificado[columnas_pacientes].values.T

# 3. Aplicamos la Imputación Multiplicativa de Ceros (usando la nueva función)
matriz_imputada_transpuesta = multi_replace(matriz_cruda_transpuesta)

# 4. Volvemos a transponer (.T)
matriz_imputada = matriz_imputada_transpuesta.T

# 5. Reconstruimos el DataFrame de Pandas
df_imputado = pd.DataFrame(
    data=matriz_imputada, 
    index=df_purificado.index, 
    columns=columnas_pacientes
)

# 6. Transformación CLR final
# Ojo: scikit-bio también espera que los pacientes sean las filas al hacer CLR
matriz_clr_transpuesta = clr(df_imputado.values.T)
matriz_clr = matriz_clr_transpuesta.T

# Tu DataFrame final, listo para la red neuronal
df_clr = pd.DataFrame(
    data=matriz_clr, 
    index=df_purificado.index, 
    columns=columnas_pacientes
)
#df_clr['Bacteria_Limpia'] = df_filtrado['Bacteria_Limpia']
print(f"Ceros restantes: {(df_imputado == 0).sum().sum()}")
print("Muestra del DataFrame con transformación CLR:")
print(df_clr.head(133))

Ceros restantes: 0
Muestra del DataFrame con transformación CLR:
                 SRR10442269_1  SRR10442270_1  SRR10442271_1  SRR10442272_1  \
Bacteria_Limpia                                                               
Abiotrophia          -1.220156      -2.048994      -2.155106      -2.252726   
Acinetobacter        -1.220156      -2.048994      -2.155106      -2.252726   
Actinomyces          -1.220156      -2.048994      -0.105348      -2.252726   
Adlercreutzia        -1.220156      -2.048994       0.003851      -0.808786   
Agathobacter         -1.220156       3.136305       2.655148       2.970847   
...                        ...            ...            ...            ...   
Turicibacter         -1.220156       1.322073       0.486703       0.009524   
Tyzzerella            4.447537       1.366525       1.778219       2.563097   
UBA1819              -1.220156      -2.048994       0.191903      -0.051101   
UCG-002              -1.220156       2.122851       2.162008      

In [58]:
# Se utilizara la funcion melt, para poder definir la relacion entre los nodos y el "peso" de la arista
# Se crea la tabla pivotante
# Se van a tener 3 columnas : Bacteria | Paciente | AbundanciaTaxonomica
# En esta etapa tmb se eliminan ceros
df_fase1 = df_clr.copy()
df_fase1 = df_fase1.reset_index()
df_fase1=df_fase1.melt(id_vars=['Bacteria_Limpia'], var_name='Paciente', value_name='AbundanciaTaxonomica')
#print(df_fase1)

#elimino los valores de 0.0
df_fase1.drop(df_fase1[df_fase1['AbundanciaTaxonomica'] == 0.0].index, inplace=True)


#df_fase1['AbundanciaTaxonomica']= np.log1p(df_fase1['AbundanciaTaxonomica'])

# hare que cada paciente este con _1
df_fase1['Paciente'] = df_fase1['Paciente'].apply(lambda x: x if x.endswith('_1') else x + '_1')

# hare que cada paciente este con _1
#df_fase1.to_csv('/content/drive/MyDrive/Tesis/desarrollo/columnasGrafo.tsv', sep='\t', index=False)


print(df_fase1)


                        Bacteria_Limpia       Paciente  AbundanciaTaxonomica
0                           Abiotrophia  SRR10442269_1             -1.220156
1                         Acinetobacter  SRR10442269_1             -1.220156
2                           Actinomyces  SRR10442269_1             -1.220156
3                         Adlercreutzia  SRR10442269_1             -1.220156
4                          Agathobacter  SRR10442269_1             -1.220156
...                                 ...            ...                   ...
18508  [Eubacterium]_xylanophilum_group  SRR10442395_1             -1.970370
18509   [Ruminococcus]_gauvreauii_group  SRR10442395_1              1.939564
18510       [Ruminococcus]_gnavus_group  SRR10442395_1              4.192233
18511      [Ruminococcus]_torques_group  SRR10442395_1              4.804901
18512                        uncultured  SRR10442395_1              1.081740

[18513 rows x 3 columns]


In [59]:
# Se configura los tensores (numeros de cada paciente y bacteria)
# como se usa pytorch, el mismo solo interpreta numeros
# se hace un diccionario de los pacientes y bacterias
# pacientes (1-126), bacterias (127-N)


pacientes = df_fase1.Paciente.unique()
diccionariopacientes= {nombre: indice for indice, nombre in enumerate(pacientes, start=1)}
print(diccionariopacientes)

bacterias = df_fase1.Bacteria_Limpia.unique()
diccionariobacterias= {nombre: indice for indice, nombre in enumerate(bacterias, start=127)}

pacientes= df_fase1.Paciente
bacterias= df_fase1.Bacteria_Limpia


df_fase1['paciente_id'] = pacientes.map(diccionariopacientes)
df_fase1['bacteria_id'] = bacterias.map(diccionariobacterias)

print(df_fase1)

len(df_fase1['Bacteria_Limpia'].unique())

# 0. Mapeamos los IDs reales a IDs posicionales locales (de 0 a N-1)
df_fase1['paciente_id'] = pd.factorize(df_fase1['paciente_id'])[0]
df_fase1['bacteria_id'] = pd.factorize(df_fase1['bacteria_id'])[0]

{'SRR10442269_1': 1, 'SRR10442270_1': 2, 'SRR10442271_1': 3, 'SRR10442272_1': 4, 'SRR10442273_1': 5, 'SRR10442274_1': 6, 'SRR10442275_1': 7, 'SRR10442276_1': 8, 'SRR10442277_1': 9, 'SRR10442278_1': 10, 'SRR10442279_1': 11, 'SRR10442280_1': 12, 'SRR10442281_1': 13, 'SRR10442282_1': 14, 'SRR10442283_1': 15, 'SRR10442284_1': 16, 'SRR10442285_1': 17, 'SRR10442286_1': 18, 'SRR10442287_1': 19, 'SRR10442288_1': 20, 'SRR10442289_1': 21, 'SRR10442290_1': 22, 'SRR10442291_1': 23, 'SRR10442292_1': 24, 'SRR10442293_1': 25, 'SRR10442294_1': 26, 'SRR10442295_1': 27, 'SRR10442296_1': 28, 'SRR10442297_1': 29, 'SRR10442298_1': 30, 'SRR10442299_1': 31, 'SRR10442300_1': 32, 'SRR10442301_1': 33, 'SRR10442302_1': 34, 'SRR10442303_1': 35, 'SRR10442304_1': 36, 'SRR10442305_1': 37, 'SRR10442306_1': 38, 'SRR10442307_1': 39, 'SRR10442308_1': 40, 'SRR10442309_1': 41, 'SRR10442310_1': 42, 'SRR10442311_1': 43, 'SRR10442312_1': 44, 'SRR10442313_1': 45, 'SRR10442314_1': 46, 'SRR10442315_1': 47, 'SRR10442316_1': 48, 

In [60]:
diagnosticos= pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/diagnosticos_pacientes.tsv', sep='\t')
# a TEA se le asignara el 1 y al HC el 0
#basandome en esto, tenemos que:

diccionarioDiagnosticos= {
    'TEA': 1,
    'HC': 0
}
diagnosticos['Diagnosis'] = diagnosticos['Diagnosis'].map(diccionarioDiagnosticos)

#luego añado la columna de diagnosticos a df_fase1 basandome en los pacientes
# es decir se hace un inner join, por cada nombre de paciente


#diagnosticos.rename(columns={'Patient Name': 'Paciente'}, inplace=True)

diagnosticos['Paciente'] = diagnosticos['Paciente'].str.strip()
df_fase1['Paciente'] = df_fase1['Paciente'].str.strip()

df_fase1['Diagnosticos'] = df_fase1['Paciente'].map(diagnosticos.set_index('Paciente')['Diagnosis'])
pacientes_con_nan = df_fase1[df_fase1['Diagnosticos'].isna()]['Paciente'].tolist()

print(df_fase1)
print(f"Total de pacientes con NaN: {len(pacientes_con_nan)}")
print(f"Ejemplo de 5 pacientes con NaN: {pacientes_con_nan}")

pacientesTEA = df_fase1[df_fase1['Diagnosticos'] == 1]
pacientesHC = df_fase1[df_fase1['Diagnosticos'] == 0]

print(len(pacientesTEA['Paciente'].unique()))

print(len(pacientesHC['Paciente'].unique()))


                        Bacteria_Limpia       Paciente  AbundanciaTaxonomica  \
0                           Abiotrophia  SRR10442269_1             -1.220156   
1                         Acinetobacter  SRR10442269_1             -1.220156   
2                           Actinomyces  SRR10442269_1             -1.220156   
3                         Adlercreutzia  SRR10442269_1             -1.220156   
4                          Agathobacter  SRR10442269_1             -1.220156   
...                                 ...            ...                   ...   
18508  [Eubacterium]_xylanophilum_group  SRR10442395_1             -1.970370   
18509   [Ruminococcus]_gauvreauii_group  SRR10442395_1              1.939564   
18510       [Ruminococcus]_gnavus_group  SRR10442395_1              4.192233   
18511      [Ruminococcus]_torques_group  SRR10442395_1              4.804901   
18512                        uncultured  SRR10442395_1              1.081740   

       paciente_id  bacteria_id  Diagno

In [61]:
#creo el dataset df_fase1 para poder usarlo en la otra notebook
df_fase1.to_csv('/kaggle/working/dataset.csv', sep='\t', index=False)